In [71]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

#df = pd.read_csv(PROCESSED_DIR / "dawn_revenue_combined_clean.csv")
#df.head()

# Load the CSV file into a pandas DataFrame
candidate_paths = [
    RAW_DIR / "dawn_revenue_accounts_combined_history.csv",
    PROJECT_ROOT.parent / "data" / "dawn_revenue_accounts_combined_history.csv",
    PROJECT_ROOT.parent / "dawn_aggregate" / "data" / "dawn_revenue_accounts_combined_history.csv",
    PROCESSED_DIR / "dawn_revenue_accounts_combined_history.csv",
]

df_all = None
for path in candidate_paths:
    if path.exists():
        df_all = pd.read_csv(path)
        break

if df_all is None:
    raise FileNotFoundError(
        "Could not find 'dawn_revenue_accounts_combined_history.csv'. "
        f"Checked: {[str(p) for p in candidate_paths]}"
    )
df_all.head()

,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,7/29/2019,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,7/30/2019,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,7/30/2019,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,7/30/2019,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,8/12/2019,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


In [72]:
print(df_all['fy'].unique())

[2020 2021 2022 2023 2024 2025 2026 2027 2002 2003 2004 2005 2006 2007
 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2000 2001]


In [73]:
# Removing fy 2027 as it is not a complete year and may skew the analysis

# df_all = df_all[df_all['fy'] != 2027]
print(f"Shape of DataFrame after dropping FY2027 data: {df_all.shape}")
display(df_all.head())

Shape of DataFrame after dropping FY2027 data: (335819, 10)


,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,7/29/2019,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,7/30/2019,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,7/30/2019,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,7/30/2019,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,8/12/2019,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


In [74]:
# Dropping GL Number 42 as it is not relevant to the analysis

df_all_filtered = df_all[df_all['GL Number'] != 42]
print(f"Shape of DataFrame after dropping GL Number 42: {df_all_filtered.shape}")
display(df_all_filtered.head())

Shape of DataFrame after dropping GL Number 42: (330146, 10)


,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,7/29/2019,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,7/30/2019,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,7/30/2019,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,7/30/2019,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,8/12/2019,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


In [75]:
# Convert Amount to numeric first
df_all['Amount'] = pd.to_numeric(df_all['Amount'], errors='coerce')

# Keep everything except GL 42 when Amount is greater than 0
df_all_filtered = df_all[
    ~(
        (df_all['GL Number'] == 42) &
        (df_all['Amount'] > 0)
    )
].copy()

print(f"Shape after dropping only GL 42 with Amount > 0: {df_all_filtered.shape}")
display(df_all_filtered.head())

Shape after dropping only GL 42 with Amount > 0: (330338, 10)


,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,7/29/2019,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,7/30/2019,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,7/30/2019,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,7/30/2019,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,8/12/2019,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


## General Fund Calculation

In [76]:
# filtering the DataFrame for BA Number 9999 and specific GL Numbers that we require for the analysis. These GL Numbers are related to the General Fund and are relevant to our analysis.

gl_numbers_to_keep = [3107, 3114, 3115, 3120, 3202, 3203, 3204, 3220, 3240, 3255, 3271, 3274, 3276, 3291, 4102, 4403, 4408, 4409]
df_filtered_ba_gl = df_all_filtered[(df_all_filtered['BA Number'] == 9999) & (df_all_filtered['GL Number'].isin(gl_numbers_to_keep))]
display(df_filtered_ba_gl.head())
print(f"Shape of DataFrame after filtering for BA Number 9999 and specific GL Numbers: {df_filtered_ba_gl.shape}")

,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
269377,CR 707 610684,1/6/2000,1114.5,2004,2653,3107,"MISC LICENSES, FEES, PERMITS",Minor,9999,2000
269378,CR 741 610838,1/7/2000,100.0,2004,2653,3107,"MISC LICENSES, FEES, PERMITS",Minor,9999,2000
269379,CR 741 610839,1/7/2000,100.0,2004,2653,3107,"MISC LICENSES, FEES, PERMITS",Minor,9999,2000
269380,CR 089 747883,1/13/2000,234.0,2004,2653,3107,"MISC LICENSES, FEES, PERMITS",Minor,9999,2000
269381,CR 707 610686,1/13/2000,1282.5,2004,2653,3107,"MISC LICENSES, FEES, PERMITS",Minor,9999,2000


Shape of DataFrame after filtering for BA Number 9999 and specific GL Numbers: (40166, 10)


In [77]:
# filtering the DataFrame for BA Number 9999 and specific GL Numbers that we require for the analysis. These GL Numbers are related to the General Fund and are relevant to our analysis.

gl_numbers_to_keep = [3107, 3114, 3115, 3120, 3202, 3203, 3204, 3220, 3240, 3255, 3271, 3274, 3276, 3291, 4102, 4403, 4408, 4409]

mask = (
    (df_all_filtered['BA Number'] != 9999) |
    (
        (df_all_filtered['BA Number'] == 9999) &
        (df_all_filtered['GL Number'].isin(gl_numbers_to_keep))
    )
)

df_filtered = df_all_filtered.loc[mask].copy()

display(df_filtered.head())
print(f"Shape after keeping all BA values and limiting only BA 9999 GL codes: {df_filtered.shape}")

,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,7/29/2019,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,7/30/2019,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,7/30/2019,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,7/30/2019,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,8/12/2019,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


Shape after keeping all BA values and limiting only BA 9999 GL codes: (298867, 10)


In [78]:
# Now I want to keep only the rows where the 'BA Nnumber' column is 9010, 9040, 9050, 9090, 9130, 9360, 9600, 9705, 9707, 9741, 9748, 9752, 9999.    
# I will filter the DataFrame accordingly.
ba_numbers_to_keep = [9010, 9040, 9050, 9090, 9130, 9360, 9600, 9705, 9707, 9741, 9748, 9752, 9999]

df_filtered_ba = df_filtered[df_filtered['BA Number'].isin(ba_numbers_to_keep)]
display(df_filtered_ba.head())
print(f"Shape of DataFrame after filtering for specific BA Numbers: {df_filtered_ba.shape}")


,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,7/29/2019,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,7/30/2019,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,7/30/2019,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,7/30/2019,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,8/12/2019,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


Shape of DataFrame after filtering for specific BA Numbers: (298867, 10)


In [79]:
import plotly.express as px

# Group by fiscal year and sum the amount
#df_gf = df_filtered_ba.groupby('fy')['Amount'].sum().reset_index()



df_gf = (
    df_filtered_ba.assign(Amount=pd.to_numeric(df_filtered_ba['Amount'], errors='coerce'))
    .groupby('fy')['Amount']
    .sum()
    .reset_index()
)


# Create an interactive line plot
fig = px.line(
    df_gf,
    x='fy',
    y='Amount',
    title='Total Amount by Fiscal Year',
    labels={'fy': 'Fiscal Year', 'Amount': 'Total Amount'}
)

fig.update_traces(mode='lines+markers')
fig.update_layout(
    hovermode='x unified',
    xaxis_title='Fiscal Year',
    yaxis_title='Total Amount',
    xaxis=dict(tickmode='linear', dtick=1) # Ensure yearly ticks
)

fig.show()

In [80]:
df_gf_table = df_filtered_ba.groupby('fy')['Amount'].sum().reset_index()
df_gf_table.columns = ['Fiscal Year', 'Total Amount']

pd.options.display.float_format = '{:,.2f}'.format
display(df_gf_table)
pd.options.display.float_format = None # Reset to default for other outputs

,Fiscal Year,Total Amount
0,2000,"12,633,446.84"
1,2001,"17,798,449.05"
2,2002,"1,758,431,755.10"
3,2003,"1,815,046,682.10"
4,2004,"2,393,262,682.67"
5,2005,"2,730,971,122.15"
6,2006,"3,044,113,849.54"
7,2007,"3,101,473,360.61"
8,2008,"2,987,809,976.33"
9,2009,"2,669,801,183.03"


In [81]:
df_2026 = df_filtered_ba[df_filtered_ba['fy'] == 2026]
df_2026_by_ba_gl = df_2026.groupby(['BA Number', 'GL Number'])['Amount'].sum().reset_index()
df_2026_by_ba_gl.columns = ['BA Number', 'GL Number', 'Total Amount']

# Format the 'Total Amount' column for better readability
pd.options.display.float_format = '{:,.2f}'.format
display(df_2026_by_ba_gl)
pd.options.display.float_format = None

,BA Number,GL Number,Total Amount
0,9010,3102,"5,997,486.03"
1,9010,3103,"165,062.10"
2,9010,3271,"22,936.62"
3,9040,3105,"3,849,325.00"
4,9040,3113,"127,042,266.00"
...,...,...,...
95,9999,3291,"643,757.92"
96,9999,4102,"3,000,000.00"
97,9999,4403,"20,670.03"
98,9999,4408,"1,036,940.68"


In [82]:
df_2027 = df_filtered_ba[df_filtered_ba['fy'] == 2027]
df_2027_by_ba_gl = df_2027.groupby(['BA Number', 'GL Number'])['Amount'].sum().reset_index()
df_2027_by_ba_gl.columns = ['BA Number', 'GL Number', 'Total Amount']

# Format the 'Total Amount' column for better readability
pd.options.display.float_format = '{:,.2f}'.format
display(df_2027_by_ba_gl)
pd.options.display.float_format = None

,BA Number,GL Number,Total Amount
0,9010,3102,"1,586,331.97"
1,9010,3103,"35,725.00"
2,9010,3271,"12,254.10"
3,9040,3105,"1,245,905.00"
4,9040,3113,"34,770,956.18"
...,...,...,...
72,9999,3274,"6,057.19"
73,9999,3276,"1,878,398.25"
74,9999,3291,"114,134.79"
75,9999,4102,"500,000.00"


In [83]:
df_2026_table = (
    df_2026.groupby(['BA Number', 'GL Number'])['Amount']
    .sum()
    .sort_index()
)

pd.options.display.float_format = '{:,.2f}'.format
display(df_2026_table)
pd.options.display.float_format = None

BA Number  GL Number
9010       3102          5,997,486.03
           3103            165,062.10
           3271             22,936.62
9040       3105          3,849,325.00
           3113        127,042,266.00
                            ...      
9999       3291            643,757.92
           4102          3,000,000.00
           4403             20,670.03
           4408          1,036,940.68
           4409            125,000.00
Name: Amount, Length: 100, dtype: float64

In [84]:
import pandas as pd
import plotly.express as px

ba_numbers_to_keep = [9010, 9040, 9050, 9090, 9130, 9360, 9600, 9705, 9707, 9741, 9748, 9752, 9999]

df_2026 = df_filtered[
    (df_filtered['fy'] == 2026) &
    (df_filtered['BA Number'].isin(ba_numbers_to_keep))
].copy()

df_2026['Amount'] = pd.to_numeric(df_2026['Amount'], errors='coerce')

plot_df = (
    df_2026.groupby(['BA Number', 'GL Number'], as_index=False)['Amount']
    .sum()
    .sort_values(['BA Number', 'GL Number'])
)

# One interactive chart per BA with values shown on each bar
for ba in sorted(plot_df['BA Number'].unique()):
    ba_df = plot_df[plot_df['BA Number'] == ba].copy()

    fig = px.bar(
        ba_df,
        x='GL Number',
        y='Amount',
        title=f'FY 2026 Total Amount by GL for BA {ba}',
        labels={'GL Number': 'GL Number', 'Amount': 'Total Amount'},
        color='Amount',
        color_continuous_scale='Viridis',
        text='Amount'
    )

    fig.update_traces(
        texttemplate='%{text:,.2f}',
        textposition='outside',
        cliponaxis=False
    )

    fig.update_layout(
        xaxis=dict(type='category'),
        template='plotly_white',
        hovermode='closest',
        yaxis_title='Total Amount',
        xaxis_title='GL Number'
    )
    fig.show()

In [85]:
import pandas as pd
import plotly.express as px

ba_numbers_to_keep = [9010, 9040, 9050, 9090, 9130, 9360, 9600, 9705, 9707, 9741, 9748, 9752, 9999]

df_2027 = df_filtered[
    (df_filtered['fy'] == 2027) &
    (df_filtered['BA Number'].isin(ba_numbers_to_keep))
].copy()

df_2027['Amount'] = pd.to_numeric(df_2027['Amount'], errors='coerce')

plot_df = (
    df_2027.groupby(['BA Number', 'GL Number'], as_index=False)['Amount']
    .sum()
    .sort_values(['BA Number', 'GL Number'])
)

# One interactive chart per BA with values shown on each bar
for ba in sorted(plot_df['BA Number'].unique()):
    ba_df = plot_df[plot_df['BA Number'] == ba].copy()

    fig = px.bar(
        ba_df,
        x='GL Number',
        y='Amount',
        title=f'FY 2027 Total Amount by GL for BA {ba}',
        labels={'GL Number': 'GL Number', 'Amount': 'Total Amount'},
        color='Amount',
        color_continuous_scale='Viridis',
        text='Amount'
    )

    fig.update_traces(
        texttemplate='%{text:,.2f}',
        textposition='outside',
        cliponaxis=False
    )

    fig.update_layout(
        xaxis=dict(type='category'),
        template='plotly_white',
        hovermode='closest',
        yaxis_title='Total Amount',
        xaxis_title='GL Number'
    )
    fig.show()

In [86]:
# General Fund major revenue by GL code for the latest fiscal year available.
last_fy = pd.to_numeric(df_filtered_ba['fy'], errors='coerce').max()
major_last_fy = df_filtered_ba.loc[
    df_filtered_ba['Type'].astype('string').str.strip().str.upper().eq('MAJOR')
    & pd.to_numeric(df_filtered_ba['fy'], errors='coerce').eq(last_fy)
].copy()
major_last_fy['Amount'] = pd.to_numeric(major_last_fy['Amount'], errors='coerce')

df_gf_major_table = (
    major_last_fy.groupby('GL Number', as_index=False)['Amount']
    .sum(min_count=1)
    .rename(columns={'Amount': 'Total Amount'})
    .sort_values('GL Number')
    .reset_index(drop=True)
)
df_gf_major_table.insert(0, 'Fiscal Year', int(last_fy))

display(
    df_gf_major_table.style
    .set_caption('General Fund: major')
    .format({'Total Amount': '{:,.2f}'})
    .hide(axis='index')
)


Fiscal Year,GL Number,Total Amount
2027,3001,"343,062,940.20"
2027,3031,"36,956,701.66"
2027,3041,"333,850,912.24"
2027,3055,"9,705,510.78"
2027,3061,"16,039,593.96"
2027,3072,"31,276,862.41"


In [87]:
# General Fund major revenue by GL code for FY2026.
major_2026 = df_filtered_ba.loc[
    df_filtered_ba['Type'].astype('string').str.strip().str.upper().eq('MAJOR')
    & pd.to_numeric(df_filtered_ba['fy'], errors='coerce').eq(2026)
].copy()
major_2026['Amount'] = pd.to_numeric(major_2026['Amount'], errors='coerce')

df_gf_major_2026_table = (
    major_2026.groupby('GL Number', as_index=False)['Amount']
    .sum(min_count=1)
    .rename(columns={'Amount': 'Total Amount'})
    .sort_values('GL Number')
    .reset_index(drop=True)
)
df_gf_major_2026_table.insert(0, 'Fiscal Year', 2026)

display(
    df_gf_major_2026_table.style
    .set_caption('General Fund: major')
    .format({'Total Amount': '{:,.2f}'})
    .hide(axis='index')
)


Fiscal Year,GL Number,Total Amount
2026,3001,"1,926,962,816.10"
2026,3031,"216,726,025.86"
2026,3041,"1,032,740,382.93"
2026,3055,"121,027,132.76"
2026,3061,"732,911,603.50"
2026,3069,"810,980,705.96"
2026,3072,"481,706,021.70"


In [88]:
# General Fund major revenue by GL code; FY2027 uses available year-to-date data.
major_years = [2025, 2026, 2027]
major_comparison = df_filtered_ba.loc[
    df_filtered_ba['Type'].astype('string').str.strip().str.upper().eq('MAJOR')
    & pd.to_numeric(df_filtered_ba['fy'], errors='coerce').isin(major_years)
].copy()
major_comparison['fy'] = pd.to_numeric(major_comparison['fy'])
major_comparison['Amount'] = pd.to_numeric(major_comparison['Amount'], errors='coerce')

df_gf_major_comparison = (
    major_comparison.groupby(['GL Number', 'fy'])['Amount']
    .sum(min_count=1)
    .unstack('fy')
    .reindex(columns=major_years)
    .sort_index()
    .rename(columns={2027: '2027 (Years-to-date)'})
    .rename_axis(index='GL Code', columns=None)
)

# Sum each fiscal-year column across all major GL codes.
df_gf_major_comparison.loc['Total'] = df_gf_major_comparison.sum(min_count=1)

display(
    df_gf_major_comparison.style
    .set_caption('General Fund: major')
    .format('{:,.2f}', na_rep='?')
)


,2025,2026,2027 (Years-to-date)
GL Code,,,
3001,"1,775,667,377.47","1,926,962,816.10","343,062,940.20"
3031,"223,110,300.34","216,726,025.86","36,956,701.66"
3041,"988,235,274.22","1,032,740,382.93","333,850,912.24"
3055,"123,785,086.16","121,027,132.76","9,705,510.78"
3061,"685,707,391.46","732,911,603.50","16,039,593.96"
3069,"781,429,893.90","810,980,705.96",?
3072,"268,130,157.33","481,706,021.70","31,276,862.41"
Total,"4,846,065,480.88","5,323,054,688.81","770,892,521.25"


In [89]:
# SQL equivalent using an in-memory SQLite database (no extra package required).
import sqlite3

sql_source = df_filtered_ba[['GL Number', 'Type', 'fy', 'Amount']].copy()
sql_source['fy'] = pd.to_numeric(sql_source['fy'], errors='coerce')
sql_source['Amount'] = pd.to_numeric(sql_source['Amount'], errors='coerce')

major_sql = """
WITH major_by_gl AS (
    SELECT
        "GL Number" AS "GL Code",
        SUM(CASE WHEN fy = 2025 THEN Amount END) AS "2025",
        SUM(CASE WHEN fy = 2026 THEN Amount END) AS "2026",
        SUM(CASE WHEN fy = 2027 THEN Amount END) AS "2027 (Years-to-date)"
    FROM general_fund
    WHERE UPPER(TRIM(Type)) = 'MAJOR'
      AND fy IN (2025, 2026, 2027)
      AND "GL Number" IS NOT NULL
    GROUP BY "GL Number"
), with_total AS (
    SELECT 0 AS sort_order, "GL Code", "2025", "2026", "2027 (Years-to-date)"
    FROM major_by_gl
    UNION ALL
    SELECT 1, 'Total', SUM("2025"), SUM("2026"), SUM("2027 (Years-to-date)")
    FROM major_by_gl
)
SELECT "GL Code", "2025", "2026", "2027 (Years-to-date)"
FROM with_total
ORDER BY sort_order, "GL Code"
"""

connection = sqlite3.connect(':memory:')
try:
    sql_source.to_sql('general_fund', connection, index=False, if_exists='replace')
    df_gf_major_sql = pd.read_sql_query(major_sql, connection).set_index('GL Code')
finally:
    connection.close()

display(
    df_gf_major_sql.style
    .set_caption('General Fund: major')
    .format('{:,.2f}', na_rep='?')
)


,2025,2026,2027 (Years-to-date)
GL Code,,,
3001,"1,775,667,377.47","1,926,962,816.10","343,062,940.20"
3031,"223,110,300.34","216,726,025.86","36,956,701.66"
3041,"988,235,274.22","1,032,740,382.93","333,850,912.24"
3055,"123,785,086.16","121,027,132.76","9,705,510.78"
3061,"685,707,391.46","732,911,603.50","16,039,593.96"
3069,"781,429,893.90","810,980,705.96",?
3072,"268,130,157.33","481,706,021.70","31,276,862.41"
Total,"4,846,065,480.88","5,323,054,688.81","770,892,521.25"


In [93]:
# Presentation table using Python's Great Tables (GT).
from great_tables import GT, loc, style

gt_data = df_gf_major_sql.reset_index().copy()
gt_data['GL Code'] = gt_data['GL Code'].astype(str)
year_columns = ['2024', '2025', '2026', '2027 (Years-to-date)']
total_rows = gt_data.index[gt_data['GL Code'].eq('Total')].tolist()

major_gt = (
    GT(gt_data)
    .tab_header(
        title='General Fund: Major (including Tax Credit) | Fiscal years 2024 -- 2027 ',
        # subtitle='Revenue by GL code | Fiscal years 2024 -- 2027',
    )
    .tab_spanner(label='Fiscal year  Amount ($)', columns=year_columns)
    .fmt_number(columns=year_columns, decimals=2, use_seps=True)
    .sub_missing(columns=year_columns, missing_text='?')
    .cols_align(align='left', columns='GL Code')
    .cols_align(align='right', columns=year_columns)
    .tab_options(
        table_width='100%',
        table_font_names=['Segoe UI', 'Arial', 'sans-serif'],
        table_font_size='14px',
        heading_align='left',
        data_row_padding='12px',
        row_striping_include_table_body=True,
        row_striping_background_color='#F5F7FA',
    )
    .tab_style(
        style=[style.fill(color='#17324D'), style.text(color='white')],
        locations=loc.header(),
    )
    .tab_style(
        style=style.text(size='24px', weight='bold'),
        locations=loc.title(),
    )
    .tab_style(
        style=[style.fill(color='#E8EEF5'), style.text(color='#17324D', weight='bold')],
        locations=loc.column_labels(),
    )
    .tab_style(
        style=[style.fill(color='#E5F3F0'), style.text(color='#12675B', weight='bold')],
        locations=loc.column_labels(columns='2027 (Years-to-date)'),
    )
    .tab_style(
        style=[
            style.fill(color='#17324D'),
            style.text(color='white', weight='bold'),
            style.borders(sides='top', color='#26A69A', weight='3px'),
        ],
        locations=loc.body(rows=total_rows),
    )
    .tab_source_note(
        source_note='2027 (Years-to-date) includes available FY2027 records only. Amounts in dollars.'
    )
)
display(major_gt)


GT(_tbl_data=  GL Code          2025          2026  2027 (Years-to-date)
0    3001  1.775667e+09  1.926963e+09          3.430629e+08
1    3031  2.231103e+08  2.167260e+08          3.695670e+07
2    3041  9.882353e+08  1.032740e+09          3.338509e+08
3    3055  1.237851e+08  1.210271e+08          9.705511e+06
4    3061  6.857074e+08  7.329116e+08          1.603959e+07
5    3069  7.814299e+08  8.109807e+08                   NaN
6    3072  2.681302e+08  4.817060e+08          3.127686e+07
7   Total  4.846065e+09  5.323055e+09          7.708925e+08, _body=<great_tables._gt_data.Body object at 0x000001966D7D0A50>, _boxhead=Boxhead([ColInfo(var='GL Code', type=<ColInfoTypeEnum.default: 1>, column_label='GL Code', column_align='left', column_width=None), ColInfo(var='2025', type=<ColInfoTypeEnum.default: 1>, column_label='2025', column_align='right', column_width=None), ColInfo(var='2026', type=<ColInfoTypeEnum.default: 1>, column_label='2026', column_align='right', column_width=None), ColInfo(var='2027 (Years-to-date)', type=<ColInfoTypeEnum.default: 1>, column_label='2027 (Years-to-date)', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001966D7722C0>, _spanners=Spanners([SpannerInfo(spanner_id='Fiscal year  Amount ($)', spanner_level=0, spanner_label='Fiscal year  Amount ($)', spanner_units=None, spanner_pattern=None, vars=['2025', '2026', '2027 (Years-to-date)'], built=None)]), _heading=Heading(title='General Fund: Major (including Tax Credit) | Fiscal years 2024 -- 2027 ', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x00000196670C2690>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x00000196670C2CF0>, _source_notes=['2027 (Years-to-date) includes available FY2027 records only. Amounts in dollars.'], _footnotes=[], _styles=[StyleInfo(locname=LocHeader(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleFill(color='#17324D'), CellStyleText(color='white', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color=None, font=None, size='24px', align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='GL Code', rownum=None, colnum=None, styles=[CellStyleFill(color='#E8EEF5'), CellStyleText(color='#17324D', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='2025', rownum=None, colnum=None, styles=[CellStyleFill(color='#E8EEF5'), CellStyleText(color='#17324D', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='2026', rownum=None, colnum=None, styles=[CellStyleFill(color='#E8EEF5'), CellStyleText(color='#17324D', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='2027 (Years-to-date)', rownum=None, colnum=None, styles=[CellStyleFill(color='#E8EEF5'), CellStyleText(color='#17324D', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns='2027 (Years-to-date)'), grpname=None, colname='2027 (Years-to-date)', rownum=None, colnum=None, styles=[CellStyleFill(color='#E5F3F0'), CellStyleText(color='#12675B', font=None, size=None, align=None, v_align=None, style=None, weight='bold',